In [ ]:
# Install dependencies
!pip install yfinance numpy pandas scipy matplotlib plotly

import yfinance as yf
import numpy as np
import pandas as pd
import scipy.stats as si
from scipy.interpolate import griddata
import plotly.graph_objects as go
from datetime import datetime

# Black-Scholes IV solver
def calculate_iv(price, S, K, T, r, flag):
    # Newton-Raphson to find IV
    sigma = 0.5
    for i in range(100):
        d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        if flag == 'call':
            f = S * si.norm.cdf(d1) - K * np.exp(-r*T) * si.norm.cdf(d2) - price
            vega = S * si.norm.pdf(d1) * np.sqrt(T)
        else:
            f = K * np.exp(-r*T) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1) - price
            vega = S * si.norm.pdf(d1) * np.sqrt(T)

        if abs(f) < 0.001: break
        sigma -= f / vega
        if sigma <= 0: return np.nan
    return sigma

In [ ]:
ticker_symbol = "SPY"
ticker = yf.Ticker(ticker_symbol)
S = ticker.history(period="1d")['Close'].iloc[-1]
r = 0.18 # Risk-free rate approximation

data = []
for expiry in ticker.options[:10]: # Take first 10 expirations
    opt = ticker.option_chain(expiry)
    calls = opt.calls
    for _, row in calls.iterrows():
        T = (datetime.strptime(expiry, "%Y-%m-%d") - datetime.now()).days / 365
        if T <= 0: # Skip options that are expired or expiring today
            continue
        iv = calculate_iv(row['lastPrice'], S, row['strike'], T, r, 'call')
        data.append({'strike': row['strike'], 'T': T, 'iv': iv})


df = pd.DataFrame(data).dropna()
print(f"Data points collected: {len(df)}")

/tmp/ipykernel_711/2977583322.py:28: RuntimeWarning: divide by zero encountered in scalar divide
  sigma -= f / vega
/tmp/ipykernel_711/2977583322.py:28: RuntimeWarning: divide by zero encountered in scalar divide
  sigma -= f / vega


Data points collected: 873


In [ ]:
# Define a grid for interpolation
strike_range = np.linspace(df['strike'].min(), df['strike'].max(), 50)
t_range = np.linspace(df['T'].min(), df['T'].max(), 50)
strike_grid, t_grid = np.meshgrid(strike_range, t_range)

# Interpolate using cubic spline
iv_grid = griddata((df['strike'], df['T']), df['iv'], (strike_grid, t_grid), method='cubic')

# Plot the 3D surface
fig = go.Figure(data=[go.Surface(z=iv_grid, x=strike_grid, y=t_grid)])
fig.update_layout(title='Implied Volatility Surface (SPY)',
                  scene=dict(xaxis_title='Strike', yaxis_title='Time to Maturity', zaxis_title='IV'))
fig.show()

**BASE** **CODE**